In [3]:
!pip install pulp

#ライブラリのインポート

import pulp 
import time
import pandas as pd
import numpy as np

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
#データの読み込み

import glob
import os
import pickle

INPUT_DIR = "/content/drive/My Drive/Google Colab/takano_Lab2/scenarios/data_7"
INPUT_CONST_DIR = "/content/drive/My Drive/Google Colab/takano_Lab2/scenarios/Constant"

#読みこみ
def pickle_load(path):
    with open(path, mode='rb') as f:
        data = pickle.load(f)
        return data

#書き出し
def pickle_dump(obj, path):
    with open(path, mode='wb') as f:
        pickle.dump(obj,f)

def import_data_dfm(y,m):
  csv_files_dir_start = sorted(glob.glob(os.path.join(INPUT_DIR, 'start/{0}/start_trip_{0}0{1}*'.format(y,m))))
  marge_csv_start = [pd.read_csv(f, encoding='cp932') for f in csv_files_dir_start]
  df_master_start = pd.concat(marge_csv_start, ignore_index=True)

  csv_files_dir_end = sorted(glob.glob(os.path.join(INPUT_DIR, 'end/{0}/end_trip_{0}0{1}*'.format(y,m))))
  marge_csv_end = [pd.read_csv(f, encoding='cp932') for f in csv_files_dir_end]
  df_master_end = pd.concat(marge_csv_end, ignore_index=True)
  sce_num = len(csv_files_dir_start)

  return df_master_start,df_master_end, sce_num

df_master_start,df_master_end, sce_num = import_data_dfm(2015,3)

station_C = pd.read_csv((os.path.join(INPUT_CONST_DIR, 'station_C.csv')))
K = pd.read_csv((os.path.join(INPUT_CONST_DIR, 'trip_duration.csv')))

In [6]:
#各種値の指定
station_num = 7
time_num = 288
scenario_num = sce_num
time_span = 1
time_start = 8
time_end = 20

#集合の生成
#ここでは8:00~20:00まで1時間ごとに計13回のリバランスを考える
D = list(range(1,station_num+1))
T = list(range(1,time_num+1))
T_ = list(range(1,time_num+2))
S = list(range(1,scenario_num+1))
T2hour = list((time*12*time_span)+1 for time in range(int(time_start/time_span),int(time_end/time_span)+1))

STJ = [(s,t,j) for j in D for t in T_ for s in S]
STIJ = [(s,t,i,j) for j in D for i in D for t in T for s in S]
TIJ = [(t,i,j) for j in D for i in D for t in T]
SJ = [(s,j) for j in D for s in S]
IJ = [(i,j) for j in D for i in D]
ST2hourIJ = [(s,t,i,j) for j in D for i in D for t in T2hour for s in S]
T2hourIJ = [(t,i,j) for j in D for i in D for t in T2hour]
ST2hourJ = [(s,t,j) for j in D for t in T2hour for s in S]

print(D)
print(T)
print(S)

s_trip = {}
e_trip = {}

for stij in STIJ:
    s_trip[stij] = 0 
    e_trip[stij] = 0

for s in S:
  n = 288*(s-1)
  for t in T:
    for i in D:
      for j in D:
        s_trip[s,t,i,j] = int(df_master_start.iloc[[n+t-1]]['{0}_{1}'.format(i-1,j-1)])         
        e_trip[s,t,i,j] = int(df_master_end.iloc[[n+t-1]]['{0}_{1}'.format(i-1,j-1)])  

[1, 2, 3, 4, 5, 6, 7]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 

In [7]:
#クラスタごとの上限値を作成
C = station_C.groupby('cluster_id')['dock_count'].agg(['sum'])
C2M = {}
i = 1
for row in C.itertuples():
    C2M[i] = row.sum
    i += 1
C2M

#駅間のトリップ時間を辞書に格納
K2D = {}

for i in D:
  for j in D:
    if i == j:
      K2D[i,j] = 0
    else:
      K2D[i,j] = K.iat[i-1,j-1]

In [8]:
def culc_trip_dif(s_trip, e_trip):
  trip_dif = {}
  trip_d = {}
  for stj in ST2hourJ:
    trip_dif[stj] = 0
    trip_d[stj] = 0
  for s in S:
    for t in T2hour:
      for j in D:
        trip_dpt = 0
        trip_arr = 0
        for n in range(t-12,t):
          for l in D:
            trip_dpt += int(s_trip[s,n,j,l])
            trip_arr += int(e_trip[s,n,l,j])
        trip_dif[s,t,j] = (trip_arr-trip_dpt)
        trip_d[s,t,j] = trip_dpt

  return trip_dif, trip_d

trip_dif,trip_d = culc_trip_dif(s_trip,e_trip)

In [19]:
#動的リバランスを考慮

class MobilitySolver:
      def __init__(self):
        pass


      def solve(self,weight_para,name):

          alpha = 1
          beta = 1 #正則化項
          gamma = weight_para
          delta = 1

          # 時間計測開始
          time_start = time.perf_counter()

          problem = pulp.LpProblem('BikeShare',pulp.LpMinimize)

          #決定変数を定義する

          z = pulp.LpVariable.dicts('z', STJ,lowBound=0, cat='Continuous')
          z_i = pulp.LpVariable.dicts('z_i', D,lowBound=0, cat='Continuous')
          a = pulp.LpVariable.dicts('a', STJ,lowBound=0, cat='Continuous')
          w = pulp.LpVariable.dicts('w', T2hourIJ,lowBound=0, cat='Continuous')
          x = pulp.LpVariable.dicts('x', SJ,lowBound=0, cat='Continuous')
          y = pulp.LpVariable.dicts('y', SJ,lowBound=0, cat='Continuous')
          if name != 'noreb':
            r_s = pulp.LpVariable.dicts('r_s', ST2hourIJ,lowBound=0, cat='Continuous')
            r = pulp.LpVariable.dicts('r', T2hourIJ,lowBound=0, cat='Continuous')
          elif name == 'noreb':
            r_s = {}
            r = {}
            for stij in ST2hourIJ:
              r_s[stij] = 0
            for tij in T2hourIJ:
              r[tij] = 0        

          def func_r_s_2hour(s,t,i,j):
            if t-K2D[i,j] in T2hour:
              t_past = t-K2D[i,j]
              return r_s[s,t_past,i,j]
            else:
              return 0

          #目的関数を宣言する
          problem += alpha*pulp.lpSum(K2D[i,j]*r_s[s,t,i,j] for j in D for i in D for t in T2hour for s in S)/scenario_num + beta*pulp.lpSum(a[s,t,j] for j in D for t in T_ for s in S)/scenario_num + gamma*(pulp.lpSum(w[t,i,j] for j in D for i in D for t in T2hour)) + delta*pulp.lpSum(x[s,j]+y[s,j] for j in D for s in S)/scenario_num


          for s in S:

            #絶対値の処理
            for j in D:
              problem += z[s,289,j]- z_i[j] == x[s,j]-y[s,j]

            problem += sum(z_i[j] for j in D) == 329

            for j in D:
              problem += z[s,1,j] == z_i[j]

            # 制約条件① 駅の車両台数の推移
            for t in T:
              for j in D:
                if t in T2hour:
                  problem += z[s,t+1,j] == z[s,t,j] - sum(s_trip[s,t,j,i] for i in D) - sum(r_s[s,t,j,i] for i in D) + sum(e_trip[s,t,i,j] for i in D)
                else:
                  problem += z[s,t+1,j] == z[s,t,j] - sum(s_trip[s,t,j,i] for i in D) + sum(e_trip[s,t,i,j] for i in D) + sum(func_r_s_2hour(s,t,i,j) for i in D)
              
            #制約条件②
            for t in T2hour:
                for i in D:
                    problem += r_s[s,t,i,i] == 0
                    problem += w[t,i,i] == 0
                    problem += r[t,i,i] == 0


            # 制約条件② 
            if name != 'streb' and name != 'noreb':
              for t in T:
                  for j in D:
                    if t in T2hour:
                      problem += z[s,t,j] >= (sum(s_trip[s,t,j,i] for i in D) + sum(r_s[s,t,j,i] for i in D))
                    else:
                      problem += z[s,t,j] >= sum(s_trip[s,t,j,i] for i in D)

            # 制約条件③ 初期値からの乖離
            for t in T_:
                for j in D:
                    problem += z[s,t,j] <= C2M[j] + a[s,t,j]
                    problem += z[s,t,j] >= 0 - a[s,t,j]

            #制約⑥　１回のリバランス量制限
            for t in T2hour:
              problem += pulp.lpSum(r_s[s,t,i,j] for i in D for j in D) <= 15

            #制約⑤　線形の動的リバランス計画
            if name == 'streb':
              for t in T2hour:
                for i in D:
                    for j in D:
                      problem += r_s[s,t,i,j] == r[t,i,j]

            elif name == 'noreb':
              for t in T2hour:
                for i in D:
                    for j in D:
                      problem += r_s[s,t,i,j] == 0
                      problem += r[t,i,j] == 0
                      problem += w[t,i,j] == 0

            elif name == '2dir':
              for t in T2hour:
                  for i in D:
                      for j in D:
                        trip_2way = 0
                        for n in range(t-12,t):
                          trip_2way += (s_trip[s,n,i,j]-s_trip[s,n,j,i])
                        problem += r_s[s,t,i,j] == r[t,i,j] - w[t,i,j]*trip_2way

            elif name == 'dpt_arr':
              for t in T2hour:
                for i in D:
                  for j in D:
                    problem += r_s[s,t,i,j] == r[t,i,j] + w[t,i,j]*(trip_dif[s,t,i]-trip_dif[s,t,j])

          status = problem.solve()
          # 時間計測終了
          time_stop = time.perf_counter()

          print(time_stop-time_start)
          print(pulp.LpStatus[status])
          print(pulp.value(problem.objective))


          Z = [[[0 for j in range(station_num)]for t in range(time_num+1)] for s in range(scenario_num)]
          A = [[[0 for j in range(station_num)]for t in range(time_num+1)] for s in range(scenario_num)]
          XY = [[0 for j in range(station_num)]for s in range(scenario_num)]

          sum_Rs = 0
          sum_Rs_K = 0
          for s in S:
              for i in D:
                  for j in D:
                      for t in T2hour:
                        if name != 'noreb':
                          sum_Rs += r_s[s,t,i,j].value()
                          sum_Rs_K += r_s[s,t,i,j].value()*K2D[i,j]
                        elif name == 'noreb':
                          sum_Rs += r_s[s,t,i,j]
                          sum_Rs_K += r_s[s,t,i,j]*K2D[i,j]

          for s in range(scenario_num):
              for j in range(station_num):
                  for t in range(time_num+1):
                      Z[s][t][j] = z[s+1,t+1,j+1].value()

          sum_A = 0
          for s in range(scenario_num):
              for j in range(station_num):
                  for t in range(time_num+1):
                      A[s][t][j] = a[s+1,t+1,j+1].value()
                      sum_A += a[s+1,t+1,j+1].value()

          sum_W = 0
          for i in D:
            for j in D:
              for t in T2hour:
                if w[t,i,j].value() == None:
                  pass
                else:
                  sum_W += w[t,i,j].value() 

          sum_R = 0
          sum_R_K = 0
          for i in D:
            for j in D:
              for t in T2hour:
                if name != 'noreb':
                  sum_R += r[t,i,j].value()
                  sum_R_K += r[t,i,j].value()*K2D[i,j]
                elif name == 'noreb':
                  sum_R += r[t,i,j]
                  sum_R_K += r[t,i,j]*K2D[i,j]                  

          sum_XY = 0
          for s in range(scenario_num):
              for j in range(station_num):
                  XY[s][j] = x[s+1,j+1].value()+y[s+1,j+1].value()
                  sum_XY += x[s+1,j+1].value()+y[s+1,j+1].value() 


          self.obj = pulp.value(problem.objective)
          self.R_s_sce = sum_Rs/scenario_num
          self.R = sum_R
          self.R_s_sce_K = sum_Rs_K/scenario_num
          self.R_K = sum_R_K         
          self.A_sce = sum_A/scenario_num
          self.XY_sce = sum_XY/scenario_num
          self.W = sum_W
          self.time = time_stop-time_start
          self.status = pulp.LpStatus[status] 

          return r, w, z, z_i, a, r_s, x, y

In [20]:
NAME_WEIGHT_LIST = [['noreb','',1],['streb','',1],
                    ['2dir','W0',0],['2dir','W0.5',0.5],['2dir','W1',1],['2dir','W1.5',1.5],['2dir','W2',2],['2dir','W2.5',2.5],['2dir','W3',3],['2dir','W3.5',3.5],['2dir','W4',4],['2dir','W4.5',4.5],['2dir','W5',5],
                    ['dpt_arr','W0',0],['dpt_arr','W0.5',0.5],['dpt_arr','W1',1],['dpt_arr','W1.5',1.5],['dpt_arr','W2',2],['dpt_arr','W2.5',2.5],['dpt_arr','W3',3],['dpt_arr','W3.5',3.5],['dpt_arr','W4',4],['dpt_arr','W4.5',4.5],['dpt_arr','W5',5]]

Rows = []
for name, weight, weight_para in NAME_WEIGHT_LIST:

  print('===', name, weight, weight_para, '===')

  ms = MobilitySolver()
  r, w, z, z_i, a, r_s, x, y = ms.solve(weight_para,name)

  row = (name, weight_para, ms.R_s_sce, ms.R_s_sce_K, ms.R, ms.R_K, ms.A_sce, ms.XY_sce, ms.W, ms.R_s_sce_K+ms.A_sce+ms.XY_sce, ms.obj, ms.time, ms.status)
  Rows.append(row)

  pickle_dump(r, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/r_{0}{1}_201503.pickle'.format(name,weight))
  pickle_dump(w, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/w_{0}{1}_201503.pickle'.format(name,weight))
  pickle_dump(z, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/z_{0}{1}_201503.pickle'.format(name,weight))
  pickle_dump(z_i, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/z_i_{0}{1}_201503.pickle'.format(name,weight))
  pickle_dump(a, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/a_{0}{1}_201503.pickle'.format(name,weight))
  pickle_dump(r_s, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/r_s_{0}{1}_201503.pickle'.format(name,weight))
  pickle_dump(x, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/x_{0}{1}_201503.pickle'.format(name,weight))
  pickle_dump(y, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/y_{0}{1}_201503.pickle'.format(name,weight))

dfm = pd.DataFrame(Rows, columns = ['name','gamma','R_s/sce','R_s*K/sce','R','R*K','A/sce','|Z[289]-Z[1]|/sce','W','R_s*K/sce+A/sce+|Z[289]-Z[1]|/sce','objective','time','status'])
dfm.to_csv('/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/dfm_201503.csv',index=None)
dfm

=== noreb  1 ===
92.15779894099978
Infeasible
3300.5483870967764
=== streb  1 ===
25.638979879000544
Optimal
639.000000000003
=== 2dir W0 0 ===
33.27654800500022
Optimal
262.11790122388754
=== 2dir W0.5 0.5 ===
32.094508614000006
Optimal
272.66282576222545
=== 2dir W1 1 ===
32.50706962100048
Optimal
282.3110145878606
=== 2dir W1.5 1.5 ===
31.622480615999848
Optimal
291.4468163507663
=== 2dir W2 2 ===
32.628206675
Optimal
300.17736044225745
=== 2dir W2.5 2.5 ===
33.0271047440001
Optimal
308.40056285977494
=== 2dir W3 3 ===
33.31251262400019
Optimal
316.05630998921447
=== 2dir W3.5 3.5 ===
33.16486612400058
Optimal
323.27873512042856
=== 2dir W4 4 ===
32.6082188270002
Optimal
329.94396902142034
=== 2dir W4.5 4.5 ===
32.84342551700047
Optimal
336.0821309441937
=== 2dir W5 5 ===
32.83762446600031
Optimal
341.7964674987749
=== dpt_arr W0 0 ===
31.91113112400035
Optimal
240.63956938033624
=== dpt_arr W0.5 0.5 ===
31.90118108600018
Optimal
243.98908971300975
=== dpt_arr W1 1 ===
31.3569535909

,name,gamma,R_s/sce,R_s*K/sce,R,R*K,A/sce,|Z[289]-Z[1]|/sce,W,R_s*K/sce+A/sce+|Z[289]-Z[1]|/sce,objective,time,status
0,noreb,1.0,0.000000,0.000000,0.000000,0.000000,3174.419355,126.129032,0.000000,3300.548387,3300.548387,92.157799,Infeasible
1,streb,1.0,69.000000,108.000000,69.000000,108.000000,429.774194,101.225806,0.000000,639.000000,639.000000,25.638980,Optimal
2,2dir,0.0,84.504308,152.312055,61.997627,100.121373,27.439069,82.366777,22.152660,262.117901,262.117901,33.276548,Optimal
3,2dir,0.5,81.619899,151.571724,60.846653,102.610609,28.301398,82.588369,20.402669,262.461492,272.662826,32.094509,Optimal
4,2dir,1.0,81.203847,150.454890,61.354286,103.872513,29.997109,83.411003,18.448014,263.863001,282.311015,32.507070,Optimal
5,2dir,1.5,80.327704,148.956957,61.307710,104.082607,31.794653,83.743880,17.967550,264.495491,291.446816,31.622481,Optimal
6,2dir,2.0,77.401768,148.038884,60.722319,107.876197,33.918131,84.666414,16.776966,266.623429,300.177360,32.628207,Optimal
7,2dir,2.5,78.290114,147.901427,63.851880,113.045914,35.309377,85.666335,15.809370,268.877139,308.400563,33.027105,Optimal
8,2dir,3.0,80.127594,148.046782,67.807063,118.101743,36.499377,86.894021,14.872043,271.440180,316.056310,33.312513,Optimal
9,2dir,3.5,80.555493,150.560543,70.922994,126.342061,36.710601,87.857626,13.757133,275.128770,323.278735,33.164866,Optimal


In [12]:
dfm

,name,gamma,R_s/sce,R_s*K/sce,R,R*K,A/sce,|Z[289]-Z[1]|/sce,W,R_s*K/sce+A/sce+|Z[289]-Z[1]|/sce,objective,time,status
0,noreb,1.0,0.290323,0.870968,0.000000,0.000000,3169.483871,126.129032,0.000000,3296.483871,3296.483871,107.925347,Infeasible
1,streb,1.0,69.000000,108.000000,69.000000,108.000000,429.774194,101.225806,0.000000,639.000000,639.000000,25.709146,Optimal
2,2dir,0.0,84.504308,152.312055,61.997627,100.121373,27.439069,82.366777,22.152660,262.117901,262.117901,33.277199,Optimal
3,2dir,0.5,81.619899,151.571724,60.846653,102.610609,28.301398,82.588369,20.402669,262.461492,272.662826,33.093384,Optimal
4,2dir,1.0,81.203847,150.454890,61.354286,103.872513,29.997109,83.411003,18.448014,263.863001,282.311015,33.559636,Optimal
5,2dir,1.5,80.327704,148.956957,61.307710,104.082607,31.794653,83.743880,17.967550,264.495491,291.446816,33.000456,Optimal
6,2dir,2.0,77.401768,148.038884,60.722319,107.876197,33.918131,84.666414,16.776966,266.623429,300.177360,33.243049,Optimal
7,2dir,2.5,78.290114,147.901427,63.851880,113.045914,35.309377,85.666335,15.809370,268.877139,308.400563,32.970800,Optimal
8,2dir,3.0,80.127594,148.046782,67.807063,118.101743,36.499377,86.894021,14.872043,271.440180,316.056310,33.598449,Optimal
9,2dir,3.5,80.555493,150.560543,70.922994,126.342061,36.710601,87.857626,13.757133,275.128770,323.278735,33.474305,Optimal
